# Wikipedia attention: Week 2

Week 2 tasks for the Wikipedia side (James). All logic lives in `src/`; this notebook only calls it and
shows the results. Run `python -m src.wiki_pipeline` first (pulls and caches the data), then
**Restart and Run All**.

**Holdout rule:** research (spike returns, edit timing, strategy) uses data before 2025-10-01 only.
The last-3-weeks check is the one place recent data is used, and only for looking.

In [1]:
import sys

sys.path.append("..")

import pandas as pd

from src import config
from src import wiki_analysis as wa
from src.trends_compare import cached_series, gdelt_daily, tracking_correlation, trendspy_weekly
from src.wiki_attention import weekly
from src.wiki_edits import daily_edits
from src.wiki_pipeline import (PROCESSED, RAW, ROOT, article_starts, load_edits, shared_path, title_revisions,
                               universe, windows)
from src.wiki_prices import research_only

pd.set_option("display.width", 160, "display.max_columns", 30)
MARKET = "US"
TRACKED = config.tickers(MARKET)          # the 10 stocks shared with Joshua
ALL = list(universe(MARKET))              # + the strategy universe (51 stocks)
HOLDOUT = pd.Timestamp(config.holdout_start())
BOOT = config.load()["wikipedia"]["bootstrap"]
WIN = windows()
OUT = ROOT / "results" / "wikipedia"
OUT.mkdir(parents=True, exist_ok=True)

## Step 1. Data coverage after the re-pull

Research views run Jul 1, 2015 to Sep 30, 2025; the recent year sits in separate `views_recent/` files.
Each article's views include all redirects to it, so renamed articles (Facebook, Inc. -> Meta Platforms,
Tesla Motors -> Tesla, Inc.) keep their history.

In [2]:
rows = []
for t in ALL:
    r = pd.read_csv(RAW / "views_research" / f"{t}.csv", index_col=0, parse_dates=True).iloc[:, 0].dropna()
    c = pd.read_csv(RAW / "views_recent" / f"{t}.csv", index_col=0, parse_dates=True).iloc[:, 0].dropna()
    revs = title_revisions(t, universe(MARKET)[t], WIN)
    starts = article_starts(revs)
    rows.append({"ticker": t, "articles": len(starts), "article_from": min(starts.values()).date(),
                 "research_first": r.index.min().date(), "research_last": r.index.max().date(),
                 "recent_first": c.index.min().date(), "recent_last": c.index.max().date(),
                 "research_views_m": r.sum() / 1e6, "zero_view_days": int((r == 0).sum()),
                 "research_edits": len(load_edits(t, universe(MARKET)[t], WIN).query("timestamp < @HOLDOUT"))})
coverage = pd.DataFrame(rows).set_index("ticker")
assert (coverage["research_last"] < HOLDOUT.date()).all() and (coverage["recent_first"] >= HOLDOUT.date()).all()
# every stock has data from Jul 1, 2015, or from the first week its company article existed
first = coverage["article_from"].clip(lower=pd.Timestamp("2015-07-01").date())
assert (coverage["research_first"] <= first + pd.Timedelta(days=6)).all()
coverage.to_csv(OUT / "coverage.csv")
print("articles starting after Jul 2015:", coverage.loc[coverage["article_from"] > pd.Timestamp("2015-07-06").date(),
                                                          ["articles", "article_from"]].to_dict("index"))
coverage.loc[TRACKED].round(1)


articles starting after Jul 2015: {'META': {'articles': 1, 'article_from': datetime.date(2019, 11, 25)}}


,articles,article_from,research_first,research_last,recent_first,recent_last,research_views_m,zero_view_days,research_edits
ticker,,,,,,,,,
AAPL,1,2015-06-29,2015-07-01,2025-09-30,2025-10-01,2026-11-01,47.6,0,3372
MSFT,1,2015-07-06,2015-07-06,2025-09-30,2025-10-01,2026-11-01,32.3,0,1958
NVDA,1,2015-07-06,2015-07-06,2025-09-30,2025-10-01,2026-11-01,13.6,0,2207
TSLA,1,2015-07-06,2015-07-06,2025-09-30,2025-10-01,2026-11-01,33.5,0,8014
AMZN,1,2015-07-06,2015-07-06,2025-09-30,2025-10-01,2026-11-01,60.5,0,3420
META,1,2019-11-25,2019-11-25,2025-09-30,2025-10-01,2026-11-01,28.3,0,1641
GOOGL,2,2015-06-29,2015-07-01,2025-09-30,2025-10-01,2026-11-01,137.8,0,5396
AMD,1,2015-06-08,2015-07-01,2025-09-30,2025-10-01,2026-11-01,8.4,0,1539
NFLX,1,2015-07-06,2015-07-06,2025-09-30,2025-10-01,2026-11-01,56.8,0,7504


In [3]:
# Wikipedia views by year for the tracked stocks (millions). META starts in late Nov 2019, when its company
# article stopped redirecting to the Facebook product article; GOOGL sums Alphabet_Inc. and Google.
yearly = pd.concat({t: pd.read_csv(RAW / "views_research" / f"{t}.csv", index_col=0, parse_dates=True).iloc[:, 0]
                    for t in TRACKED}, axis=1)
(yearly.groupby(yearly.index.year).sum() / 1e6).round(2)


,AAPL,MSFT,NVDA,TSLA,AMZN,META,GOOGL,AMD,NFLX,JPM
date,,,,,,,,,,
2015,2.65,1.44,0.29,1.40,2.02,0.00,10.12,0.33,1.90,0.51
2016,4.98,2.97,0.67,3.17,4.13,0.00,14.04,0.65,3.67,1.14
2017,4.51,2.71,0.79,3.31,5.56,0.00,12.10,0.80,3.41,1.15
2018,4.81,2.65,0.78,3.25,5.42,0.00,10.58,0.82,4.29,1.21
2019,4.73,3.01,0.65,2.89,5.45,0.18,11.79,0.82,5.24,1.25
2020,5.03,3.27,1.07,3.64,6.88,2.55,12.30,1.13,6.74,1.16
2021,4.47,3.76,0.96,4.23,9.73,6.73,15.57,0.80,8.15,1.16
2022,4.66,4.08,1.04,3.68,8.88,8.44,18.89,0.74,9.38,1.13
2023,5.07,3.29,1.68,3.02,5.42,6.07,15.20,0.72,6.22,1.38


### Matching stocks to articles

`python -m src.wiki_titles` checks every title in `config.yaml`: the page exists, is the article itself
(not a redirect or a disambiguation page) and its Wikidata item lists the ticker. This cell needs the network.


In [4]:
from src.wiki_titles import check

titles = check(MARKET)
assert titles["ok"].all(), titles[~titles["ok"]]
print(len(titles), "titles ok; ticker missing on Wikidata:", titles.loc[~titles["ticker_listed"], "title"].tolist())
titles[titles["ticker"].isin(["GOOGL", "META"])]


52 titles ok; ticker missing on Wikidata: ['Netflix']


,market,ticker,lang,title,exists,redirect_to,disambiguation,wikidata,ticker_listed,ok
5,US,META,en,Meta_Platforms,True,None,False,Q380,True,True
6,US,GOOGL,en,Alphabet_Inc.,True,None,False,Q20800404,True,True
7,US,GOOGL,en,Google,True,None,False,Q95,True,True


## Step 2. Spike flags, past weeks only

`flag_spikes` now takes the robust z-score's median and MAD from earlier weeks only (26 needed), the same
rule as Joshua's Reddit side (`config.yaml` `spike`). The look-ahead test
(`tests/test_wiki_attention.py::test_changing_future_weeks_leaves_past_flags_unchanged`) changes only weeks
after t and checks every flag up to t is unchanged.

In [5]:
att_r = pd.read_csv(shared_path("research"))
att_c = pd.read_csv(shared_path("recent"))
assert pd.to_datetime(att_r["week"]).max() + pd.Timedelta(days=6) < HOLDOUT
spike_counts = (att_r[att_r["ticker"].isin(TRACKED)].groupby(["ticker", "source"])["spike"].agg(["sum", "count"])
                .unstack("source"))
spike_counts

sum                           count                
source wikipedia_edits wikipedia_views wikipedia_edits wikipedia_views
ticker                                                                
AAPL                13              21             534             534
AMD                  5              10             534             534
AMZN                13               9             534             534
GOOGL               17              23             534             534
JPM                 11               7             534             534
META                11              24             305             305
MSFT                10              13             534             534
NFLX                 9              15             534             534
NVDA                 9              37             534             534
TSLA                22              32             534             534

## Step 3. Returns 1-4 weeks after a spike

Spike weeks vs all other weeks, research period only. `ret_next{k}` runs from the last close of the spike
week to k weeks later; abnormal returns subtract beta x SPY (52-week beta from earlier weeks).
95% CIs resample whole weeks (1,000 times), because stocks move together within a week.

In [6]:
prices = pd.read_csv(PROCESSED / "weekly_prices.csv", parse_dates=["week"])
prices_r = research_only(prices, HOLDOUT)
att_rd = att_r.assign(week=pd.to_datetime(att_r["week"]))
spike_ret = pd.concat([
    wa.spike_returns(att_rd, prices_r, "wikipedia_views", TRACKED, **BOOT).assign(universe="10 tracked"),
    wa.spike_returns(att_rd, prices_r, "wikipedia_views", None, **BOOT).assign(universe="51 stocks"),
    wa.spike_returns(att_rd, prices_r, "wikipedia_edits", None, **BOOT).assign(universe="51 stocks"),
], ignore_index=True)
spike_ret.to_csv(OUT / "spike_returns.csv", index=False)
show = spike_ret.copy()
for c in ["mean_spike", "mean_normal", "diff", "ci_low", "ci_high"]:
    show[c] = (100 * show[c]).round(2)   # percent
show[["universe", "source", "value", "n_spike", "n_normal", "mean_spike", "mean_normal", "diff", "ci_low", "ci_high"]]

,universe,source,value,n_spike,n_normal,mean_spike,mean_normal,diff,ci_low,ci_high
0,10 tracked,wikipedia_views,ret_next1,191,4910,1.18,0.69,0.49,-0.41,1.38
1,10 tracked,wikipedia_views,ret_next2,191,4900,2.11,1.39,0.73,-0.49,1.91
2,10 tracked,wikipedia_views,ret_next3,190,4891,2.57,2.11,0.46,-1.09,2.15
3,10 tracked,wikipedia_views,ret_next4,190,4881,3.32,2.84,0.48,-1.73,2.77
4,10 tracked,wikipedia_views,abn_ret_next1,191,4910,0.47,0.35,0.13,-0.60,0.89
5,10 tracked,wikipedia_views,abn_ret_next2,191,4900,0.98,0.68,0.29,-0.79,1.44
6,10 tracked,wikipedia_views,abn_ret_next3,190,4891,1.06,1.04,0.02,-1.27,1.53
7,10 tracked,wikipedia_views,abn_ret_next4,190,4881,1.12,1.39,-0.27,-2.23,1.56
8,51 stocks,wikipedia_views,ret_next1,702,26252,0.62,0.38,0.24,-0.13,0.62
9,51 stocks,wikipedia_views,ret_next2,701,26202,1.20,0.76,0.44,-0.09,0.98


## Step 5. Edit-summary analysis

Per company per week: edits, distinct editors, bytes added/removed, reverts and keyword tags
(keywords in `config.yaml` `wikipedia.edit_tags`).

In [7]:
we_r = pd.read_csv(PROCESSED / "weekly_edits_research.csv", parse_dates=["week"])
tag_cols = [c for c in we_r.columns if c.startswith("tag_")]
per_week = we_r[we_r["ticker"].isin(TRACKED)].groupby("ticker")[["edits", "editors", "reverts", *tag_cols]].mean()
per_week.round(2)

,edits,editors,reverts,tag_earnings,tag_ceo_executive,tag_legal,tag_acquisition_merger,tag_product_launch,tag_layoff,tag_stock_shares
ticker,,,,,,,,,,
AAPL,6.31,3.81,0.41,0.14,0.08,0.04,0.06,0.17,0.01,0.06
AMD,2.88,1.97,0.42,0.01,0.01,0.01,0.10,0.13,0.00,0.02
AMZN,6.40,4.34,0.66,0.04,0.07,0.05,0.05,0.15,0.00,0.02
GOOGL,10.06,6.80,1.77,0.08,0.13,0.19,0.10,0.17,0.01,0.05
JPM,3.76,2.10,0.40,0.01,0.04,0.05,0.05,0.01,0.00,0.01
META,5.38,3.70,0.74,0.20,0.10,0.16,0.09,0.04,0.11,0.10
MSFT,3.66,2.44,0.31,0.03,0.05,0.01,0.20,0.04,0.03,0.02
NFLX,14.05,9.72,2.62,0.04,0.03,0.08,0.11,0.23,0.01,0.04
NVDA,4.13,2.62,0.52,0.02,0.02,0.02,0.26,0.14,0.00,0.04


In [8]:
# Which tags show up more in view-spike weeks? (lift = tagged edits per week in spike weeks / other weeks)
lift = wa.tag_lift(we_r, att_rd)
lift.to_csv(OUT / "tag_lift.csv")
lift.round(2)

,normal_weeks,spike_weeks,lift,total
tag_layoff,0.00,0.02,4.72,107
tag_ceo_executive,0.03,0.14,4.03,996
tag_acquisition_merger,0.07,0.23,3.43,1940
tag_stock_shares,0.02,0.06,2.60,645
tag_legal,0.03,0.09,2.51,968
tag_product_launch,0.06,0.14,2.38,1634
tag_earnings,0.02,0.05,2.35,574
edits,3.25,7.56,2.33,90825
reverts,0.46,0.90,1.96,12738


In [9]:
# Tone of edit summaries (VADER, as in Joshua's scorers)
edits_tracked = {t: load_edits(t, universe(MARKET)[t], WIN) for t in TRACKED}
research_edits = pd.concat([e[e["timestamp"] < HOLDOUT] for e in edits_tracked.values()])
tone = wa.summary_tone(research_edits["comment"])
print({k: round(v, 3) for k, v in tone.items()})
# Week 3: the shared FinBERT scorer on the same summaries; `agree` = share of summaries both label the same
both = wa.summary_tone_compare(research_edits["comment"], PROCESSED / "finbert_cache.csv")
both.to_csv(OUT / "summary_tone_vader_finbert.csv")
both.round(3)

{'n_summaries': 29796, 'share_empty': 0.196, 'share_neutral': 0.725, 'share_positive': 0.167, 'share_negative': 0.109}


/Users/jamesmulyono/Desktop/DS3_Indicator_Lab/indicatorlab/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,share_neutral,share_positive,share_negative,overall,n_summaries
vader,0.725,0.167,0.109,NaN,29796
finbert,0.924,0.024,0.052,NaN,29796
agree,0.947,0.052,0.195,0.716,29796


In [10]:
# Timing: edit spikes around view spikes and big price moves (lag < 0: edits came first)
timing = pd.concat([wa.edit_timing(att_rd[att_rd["ticker"].isin(TRACKED)], prices_r).assign(universe="10 tracked"),
                    wa.edit_timing(att_rd, prices_r).assign(universe="51 stocks")])
timing.to_csv(OUT / "edit_timing.csv", index=False)
timing.pivot_table(index=["universe", "event", "signal"], columns="lag_weeks", values="lift").round(2)

lag_weeks                                -2    -1     0     1     2
universe   event          signal                                   
10 tracked big price move edits spike  0.49  0.97  2.57  0.80  1.45
                          views spike  1.13  1.22  1.92  1.21  1.11
           views spike    edits spike  1.34  2.01  3.34  1.56  1.34
51 stocks  big price move edits spike  1.09  1.09  1.80  1.14  1.04
                          views spike  1.22  1.57  3.14  1.77  1.44
           views spike    edits spike  1.47  2.46  3.92  1.21  0.95

In [11]:
# Within a view-spike week: day of most edits minus day of most views
lags = []
for t in TRACKED:
    v = pd.read_csv(RAW / "views_research" / f"{t}.csv", index_col=0, parse_dates=True).iloc[:, 0]
    e = edits_tracked[t]
    d = daily_edits(e[e["timestamp"] < HOLDOUT], v.index)
    wk = pd.to_datetime(att_r[(att_r["ticker"] == t) & (att_r["source"] == "wikipedia_views") & att_r["spike"]]["week"])
    lags.append(wa.daily_peak_lag(v, d, list(wk)))
lags = pd.concat(lags)
print(len(lags), "view-spike weeks with edits nearby; median lag", lags.median(), "days")
lags.value_counts().sort_index()

190 view-spike weeks with edits nearby; median lag 0.0 days


edit_peak_minus_view_peak_days
-7.000000    13
-6.000000     7
-5.000000     5
-4.500000     1
-4.000000     4
-3.333333     1
-3.000000    11
-2.666667     1
-2.250000     1
-2.000000     8
-1.800000     1
-1.666667     1
-1.500000     4
-1.000000    14
-0.500000     4
-0.400000     1
 0.000000    21
 0.444444     1
 0.666667     1
 1.000000    10
 1.250000     1
 1.500000     2
 1.666667     1
 2.000000    12
 2.166667     1
 2.333333     1
 2.666667     1
 3.000000    10
 3.500000     2
 4.000000    15
 4.500000     2
 5.000000     8
 6.000000    10
 6.500000     2
 7.000000    12
Name: count, dtype: int64

### Edit sentiment (FinBERT)

Each week's net change to the article (the revision live at Monday 00:00 vs the one live at the end of
the week) gives the sentences editors added; FinBERT (`ProsusAI/finbert`) labels each one positive,
negative or neutral. Scored by the scorer shared with Reddit (`src/sentiment/finbert_scorer.py`), so both
report `sentiment_pos/neg/neu`; a week's `sentiment_net` is mean P(positive) minus mean P(negative).
Built by `python -m src.wiki_sentiment` (tracked stocks, research period).


In [12]:
from src.wiki_sentiment import analyse

sent = analyse("research")
print(pd.Series(sent["summary"]).round(3).to_string())
r = sent["returns"]
(r.set_index(["comparison", "value"])[["n_spike", "n_normal", "diff", "ci_low", "ci_high"]] * [1, 1, 100, 100, 100]).round(2)


ticker_weeks_with_edits     4553.000
ticker_weeks_scored         3628.000
sentences                  32672.000
share_neutral                  0.692
share_negative                 0.164
share_positive                 0.144
share_stale                    0.284
share_negative_stale           0.145
share_negative_fresh           0.172
corr_net_same_week             0.005
corr_net_next_week            -0.013


n_spike  n_normal  diff  ci_low  ci_high
comparison                              value                                                  
negative edits vs rest                  abn_ret           1073      2555  0.00   -0.32     0.34
                                        abn_ret_next1     1069      2553 -0.10   -0.40     0.22
                                        abn_ret_next2     1069      2546 -0.22   -0.66     0.21
                                        abn_ret_next3     1067      2538 -0.05   -0.69     0.52
                                        abn_ret_next4     1066      2534  0.20   -0.48     0.88
positive edits vs rest                  abn_ret           1069      2559 -0.03   -0.37     0.29
                                        abn_ret_next1     1068      2554  0.04   -0.31     0.38
                                        abn_ret_next2     1065      2550 -0.10   -0.62     0.39
                                        abn_ret_next3     1061      2544 -0.10   -0.73     0.50
                                        abn_ret_next4     1059      2541 -0.25   -1.00     0.40
view spikes: negative vs positive edits abn_ret             67        43 -1.49   -4.00     0.90
                                        abn_ret_next1       67        43  0.82   -1.18     2.81
                                        abn_ret_next2       67        43  1.43   -1.32     4.19
                                        abn_ret_next3       67        43 -0.28   -3.39     2.73
                                        abn_ret_next4       67        43  0.28   -3.25     3.85

## Step 6. Last-3-weeks spike check (looking only)

Weeks starting Sep 28, Oct 5 and Oct 12, 2026. Wikipedia views and edits columns for the table Joshua
builds (his `spike_check_recent.py` joins on ticker and week), plus the price move that week and the next.

In [13]:
LAST_WEEKS = ["2026-09-28", "2026-10-05", "2026-10-12"]
last3 = wa.last_weeks_table(att_c, prices.assign(week=prices["week"].dt.strftime("%Y-%m-%d")), LAST_WEEKS, TRACKED)
last3.to_csv(OUT / "last3weeks_wikipedia.csv", index=False)
show = last3.copy()
for c in ["ret_week", "ret_next_week", "abn_ret_week"]:
    show[c] = (100 * show[c]).round(1)
show.round(2)

,ticker,market,week,wiki_views,wiki_views_ratio,wiki_views_rz,wiki_views_spike,wiki_edits,wiki_edits_ratio,wiki_edits_rz,wiki_edits_spike,ret_week,ret_next_week,abn_ret_week
0,AAPL,US,2026-09-28,45188.0,0.89,-0.88,False,1.0,0.50,-0.92,False,-2.2,0.9,-2.0
1,AAPL,US,2026-10-05,43447.0,0.85,-1.21,False,3.0,1.14,0.18,False,0.9,2.0,-0.1
2,AAPL,US,2026-10-12,41474.0,0.81,-1.59,False,1.0,0.50,-0.92,False,2.0,1.2,3.6
3,AMD,US,2026-09-28,14028.0,1.53,2.96,True,2.0,1.00,0.00,False,0.5,-4.1,0.9
4,AMD,US,2026-10-05,12078.0,1.31,1.93,False,5.0,1.71,0.52,False,-4.1,-13.0,-6.2
5,AMD,US,2026-10-12,11276.0,1.21,1.33,False,0.0,0.25,-1.35,False,-13.0,-2.3,-8.3
6,AMZN,US,2026-09-28,55538.0,0.98,-0.09,False,2.0,0.86,-0.20,False,0.7,4.3,1.1
7,AMZN,US,2026-10-05,57995.0,1.02,0.30,False,1.0,0.67,-0.54,False,4.3,-4.7,2.3
8,AMZN,US,2026-10-12,57837.0,1.02,0.27,False,3.0,1.60,0.62,False,-4.7,-0.8,-1.1
9,GOOGL,US,2026-09-28,160242.0,1.09,0.90,False,1.0,0.11,-3.61,False,-0.1,2.4,0.2


In [14]:
# What the editors wrote in spiking stock-weeks (to read before writing the 2-3 sentences per stock)
spiking = last3[last3["wiki_views_spike"] | last3["wiki_edits_spike"]]
for r in spiking.itertuples():
    w = pd.Timestamp(r.week)
    e = edits_tracked[r.ticker]
    e = e[(e["timestamp"] >= w) & (e["timestamp"] < w + pd.Timedelta(days=7)) & ~e["revert"]]
    print(f"\n{r.ticker} week of {r.week}: views {r.wiki_views:.0f} ({r.wiki_views_ratio:.1f}x), "
          f"edits {r.wiki_edits:.0f} ({r.wiki_edits_ratio:.1f}x)")
    for c in e["comment"].str.slice(0, 110).loc[lambda s: s.str.len() > 0].drop_duplicates().head(6):
        print("   -", c)


AMD week of 2026-09-28: views 14028 (1.5x), edits 2 (1.0x)
   - /* Acquisition history */ World Labs

TSLA week of 2026-10-12: views 44799 (2.1x), edits 4 (0.8x)
   - Updated 2nd-gen Roadster with Oct 2026 re-reveal (TechCrunch, Jalopnik); copyedits (grammar, capitalization, m
   - /* Tesla next-generation vehicle */ removed double link in subsection
   - ce
   - Marked a citation as subscription-restricted


## Step 7. Google Trends alternatives

How well each free source tracks AAPL's weekly Wikipedia views (research weeks). trendspy is used for
exploring only; GDELT is free but rate-limited (one request per 5 seconds), so results are cached.

In [15]:
TRENDS = RAW.parent / "trends"
aapl_views = pd.read_csv(RAW / "views_research" / "AAPL.csv", index_col=0, parse_dates=True).iloc[:, 0]
track = {}
gt = cached_series(TRENDS / "trendspy_AAPL_Apple.csv",
                   lambda: trendspy_weekly("Apple", pd.Timestamp("2020-10-01"), pd.Timestamp("2025-09-28")))
track["Google Trends via trendspy ('Apple')"] = tracking_correlation(gt, aapl_views)
try:
    gd = cached_series(TRENDS / "gdelt_AAPL.csv",
                       lambda: gdelt_daily('("Apple Inc" OR "Tim Cook" OR iPhone)',
                                           pd.Timestamp("2017-01-02"), pd.Timestamp("2025-09-28")))
    track["GDELT news volume"] = tracking_correlation(weekly(gd), aapl_views)
except RuntimeError as err:   # GDELT refuses bursts of requests
    print("GDELT not available right now:", str(err)[:80])
pd.DataFrame(track).T.round(2)

,weeks,corr_level,corr_change,corr_level_spearman
Google Trends via trendspy ('Apple'),261.0,0.2,0.59,0.12
GDELT news volume,454.0,0.2,0.54,0.25


In [16]:
alternatives = pd.DataFrame([
    ["Wikipedia page views", "yes (CC0 data, documented API)", "yes, Jul 2015", "yes: hi/zh Wikipedia, but smaller audiences", "free", "the reference"],
    ["GDELT news volume and tone", "yes (open data)", "yes, Jan 2017 (DOC API)", "yes, many countries and languages", "free, rate-limited", "see table above"],
    ["trendspy", "no: scrapes Google's site", "yes, 2004 (weekly only in <=5-year windows)", "yes", "free", "see table above"],
    ["SerpApi Google Trends", "yes (paid API)", "yes, 2004", "yes", "from about $25/month for 1,000 searches", "same data as trendspy"],
    ["Glimpse", "yes (paid)", "yes", "yes", "no public API price (tool from ~$99/month)", "modelled 'absolute' volumes"],
], columns=["option", "legal to use", "history before 2023", "India and Hong Kong", "cost", "tracks AAPL views"])
alternatives

,option,legal to use,history before 2023,India and Hong Kong,cost,tracks AAPL views
0,Wikipedia page views,"yes (CC0 data, documented API)","yes, Jul 2015","yes: hi/zh Wikipedia, but smaller audiences",free,the reference
1,GDELT news volume and tone,yes (open data),"yes, Jan 2017 (DOC API)","yes, many countries and languages","free, rate-limited",see table above
2,trendspy,no: scrapes Google's site,"yes, 2004 (weekly only in <=5-year windows)",yes,free,see table above
3,SerpApi Google Trends,yes (paid API),"yes, 2004",yes,"from about $25/month for 1,000 searches",same data as trendspy
4,Glimpse,yes (paid),yes,yes,no public API price (tool from ~$99/month),modelled 'absolute' volumes


## Step 11. Strategy: Wiki Attention Breakout (Wikipedia-only)

Rules fixed before the first run (`src/wiki_strategy.py`): views spike + edit burst (>= 2x the 8-week
median) + price above its 50-day average; buy at the next day's close; exit after 4 weeks, an 8% fall,
or a close below the 50-day average; equal weight, max 10% per stock, 50% exposure when SPY is below
its 200-day average; 10 bps costs. 51 stocks, team split (Week 3): train Jun 2016 to Dec 2021, test Jan 2022 to Sep 2025, 200 random
portfolios.

In [17]:
from src.wiki_strategy import run

res = run(MARKET)
table = pd.DataFrame({k: res[k] for k in ("train", "test")}).T
print(res["decision"], "-", res["why"])
table[["n_trades", "cagr", "sharpe", "max_drawdown", "turnover", "index_cagr", "index_sharpe", "ew_cagr",
       "ew_sharpe", "random_median_sharpe", "pct_random_beaten"]].astype(float).round(3)

Keep on watch - beat >= 90% of random portfolios in test only


,n_trades,cagr,sharpe,max_drawdown,turnover,index_cagr,index_sharpe,ew_cagr,ew_sharpe,random_median_sharpe,pct_random_beaten
train,83.0,-0.006,-0.241,-0.066,3.151,0.179,0.999,0.279,1.376,0.235,5.5
test,55.0,0.055,1.561,-0.031,3.014,0.111,0.664,0.127,0.760,0.522,100.0


In [18]:
res["trades"]["reason"].value_counts()

reason
time             79
stop_or_trend    60
Name: count, dtype: int64

## Findings

See `docs/wikipedia/week2_report.md` for the written summary of these results.